# DeepClean
> Data Cleaning Tool

**This tool handles:**
* Missing Values
* Outliers
* Categorical Encoding
* Date Features
* Normalization / Standardization
* Duplicates

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from datetime import datetime, timedelta
import random
from scipy import stats
import gc

print("Libraries imported successfully!")

Libraries imported successfully!


## Data Generation Function

Uses vectorized numpy operations to ensure it can scale to millions of rows quickly without relying on slow pandas `.apply()` methods during the base generation.

In [2]:
def generate_user_data(n_rows=1_000_000):
    print(f"Generating {n_rows} base rows for user schema...")
    
    np.random.seed(42)
    
    # 1. Age
    age = np.random.normal(loc=35.0, scale=12.0, size=n_rows)
    age = np.clip(age, 18, 90).astype(int)
    
    # 2. Income
    income = np.random.lognormal(mean=10.5, sigma=0.8, size=n_rows)
    income = np.round(income, 2)
    
    # 3. City
    cities = ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix', 'Philadelphia', 'San Antonio']
    city_probs = [0.25, 0.20, 0.15, 0.15, 0.10, 0.10, 0.05]
    city_data = np.random.choice(cities, size=n_rows, p=city_probs)
    
    # 4. Joined Date
    base_date = np.datetime64('2015-01-01')
    random_days = np.random.randint(0, 3000, size=n_rows)
    joined_date = base_date + random_days.astype('timedelta64[D]')
    
    # 5. Notes
    phrase_pool = [
        "Customer complained about service.",
        "Requested a refund.",
        "VIP account upgraded.",
        "Left a 5-star review.",
        "Inactive for 6 months.",
        "Password reset required.",
        "Pending address verification.",
        "No issues reported."
    ]
    notes = np.random.choice(phrase_pool, size=n_rows)
    
    df = pd.DataFrame({
        'age': age,
        'income': income,
        'city': city_data,
        'joined_date': joined_date,
        'notes': notes
    })

    print("Injecting anomalies...")

    # --- MISSING VALUES ---
    df.loc[np.random.choice(df.index, size=int(n_rows * 0.08), replace=False), 'age'] = np.nan
    df.loc[np.random.choice(df.index, size=int(n_rows * 0.12), replace=False), 'income'] = np.nan
    df.loc[np.random.choice(df.index, size=int(n_rows * 0.50), replace=False), 'notes'] = np.nan

    # --- OUTLIERS ---
    age_outlier_idx = np.random.choice(df.index, size=int(n_rows * 0.005), replace=False)
    df.loc[age_outlier_idx, 'age'] = np.random.choice([-15, -5, 150, 999], size=len(age_outlier_idx))
    
    inc_outlier_idx = np.random.choice(df.index, size=int(n_rows * 0.005), replace=False)
    df.loc[inc_outlier_idx, 'income'] = df.loc[inc_outlier_idx, 'income'] * np.random.uniform(50, 500, size=len(inc_outlier_idx))
    df.loc[np.random.choice(df.index, size=int(n_rows * 0.001), replace=False), 'income'] = -5000.00

    # --- MESSY CATEGORICALS ---
    messy_city_idx = np.random.choice(df.dropna(subset=['city']).index, size=int(n_rows * 0.07), replace=False)
    df.loc[messy_city_idx, 'city'] = df.loc[messy_city_idx, 'city'].apply(
        lambda x: f"  {str(x).lower()} " if np.random.rand() > 0.5 else f"{str(x).upper()}   "
    )

    # --- DUPLICATES ---
    duplicates = df.sample(frac=0.03, random_state=42)
    df = pd.concat([df, duplicates], ignore_index=True)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"Generation complete. Final shape: {df.shape}")
    return df

## Execution and Memory Management

**Note on Memory Constraints:** Adjust `n_rows` carefully. If your machine freezes, it is likely due to the Jupyter kernel retaining data copies in memory. Use `gc.collect()` if doing multiple runs.

In [3]:
# Force garbage collection to clear ghost references before generation
gc.collect() 

# WARNING: Keep n_rows low (e.g., 10,000) for logic testing in notebooks. 
# Attempting 10,000,000 here will likely crash your kernel.
df = generate_user_data(n_rows=10_000)

Generating 10000 base rows for user schema...
Injecting anomalies...
Generation complete. Final shape: (10300, 5)


In [4]:
df.head()

,age,income,city,joined_date,notes
0,43.0,32984.15,Houston,2017-01-26,NaN
1,41.0,160169.82,new york,2017-01-18,Inactive for 6 months.
2,51.0,101517.23,Chicago,2022-12-27,Customer complained about service.
3,40.0,20816.13,Phoenix,2016-07-10,NaN
4,26.0,37968.19,New York,2016-09-16,NaN


## Verify Targets

In [ ]:
print("--- Data Info ---")
display(df.info())

print("\n--- Missing Values ---")
display(df.isnull().sum())

print("\n--- Outlier Check (Age) ---")
display(df['age'].describe())

print("\n--- Outlier Check (Income) ---")
display(df['income'].describe())

## Tool Definition

In [ ]:
class DeepClean:
    def _init_(self, df):
        self.df = df.copy()  # pyright: ignore[reportUnknownMemberType]
        self.report = {}

    def remove_duplicates(self):
        before = len(self.df)
        self.df = self.df.drop_duplicates()
        after = len(self.df)
        self.report["duplicates_removed"] = before - after
